# DDPM на CIFAR-10: подробная лабораторная на русском

**Цель:** понять, как изображение превращается в шум, чему учится нейросеть и как из нового шума получить изображение. Здесь реализованы две учебные модели: MLP и простая свёрточная сеть.

Это продолжение проекта [Lecture 1 / Lab One](lecture_01/README.md). Lab One знакомит с численным моделированием ОДУ/СДУ; здесь добавляется обучение нейросети. После выполнения вы должны уметь объяснить каждую часть формулы обратного шага и отличить ошибку реализации от недостаточного обучения.

## Как проходить

1. Создайте окружение по [README](README.md). Для Colab загрузите этот ноутбук и при необходимости выполните `%pip install torch torchvision numpy matplotlib` в отдельной ячейке.
2. Выполните этапы 0–3: импорты, загрузка данных, расписание и прямой процесс.
3. Прочитайте формулу потерь и обучите MLP. Свёрточный вариант — отдельный эксперимент; можно выполнить оба последовательно.
4. В ячейке выбора контрольной точки установите `MODEL_KIND='mlp'` или `'conv'` и загрузите соответствующие обученные веса.
5. Выполните обратный шаг, цикл и показ изображений. Сохраните параметры и результаты, затем меняйте один параметр за раз.

Для каждой ячейки кода: **Shift+Enter**. Определение функции не запускает эксперимент. После изменения расписания или модели повторите все зависящие от них ячейки. Обучение на полном CIFAR-10 может быть длительным, особенно на CPU. Короткий запуск проверяет работоспособность, но не качество генерации.

**Примечание о результатах:** прежние сохранённые выводы очищены при исправлении кода, чтобы они не выглядели результатами обновлённого запуска. Численные функции и один шаг обучения проверяются отдельно на синтетических данных; полноценные модели нужно обучить.

## Обозначения

| Математика | Код | Значение |
|---|---|---|
| $x_0$ | `x_0` | Чистое изображение в диапазоне [-1, 1] |
| $x_t$ | `x_t` | Зашумлённая картинка; диапазон уже не ограничен [-1, 1] |
| $\beta_t$ | `betas` | Дисперсия шума одного прямого перехода |
| $\alpha_t=1-\beta_t$ | `alphas` | Коэффициент сохранения сигнала |
| $\bar\alpha_t=\prod_{s=1}^t\alpha_s$ | `alphas_cumprod` | Накопленное сохранение сигнала |
| $\epsilon$ | `noise` | Известный стандартный нормальный шум при обучении |
| $\epsilon_\theta(x_t,t)$ | `noise_pred` | Шум, предсказанный сетью |
| $B$ | `batch_size` | Число картинок в пачке |

**Время:** в формулах статьи используются уровни $t=1,\ldots,T$. Индекс Python `k=0,...,T-1` соответствует математическому уровню $t=k+1$. Поэтому `q_sample(..., t=0)` уже добавляет первый небольшой шум, а условие `t != 0` в обратном процессе отключает случайность последнего перехода к чистому изображению.

Основной первоисточник — [Ho, Jain, Abbeel, 2020](https://arxiv.org/abs/2006.11239); дополнительное объяснение — [Lilian Weng, What are Diffusion Models?](https://lilianweng.github.io/posts/2021-07-11-diffusion-models/).

## Этап 0. Библиотеки и воспроизводимость

PyTorch выполняет расчёты и обучает сети; torchvision загружает изображения; Matplotlib рисует результаты. `seed` повторяет псевдослучайный эксперимент в одном окружении, но не обещает побитового совпадения между CPU, GPU и версиями библиотек.

In [ ]:
# Основные библиотеки: тензоры, нейросети, функции потерь, данные и графики.
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import numpy as np
import matplotlib.pyplot as plt

# Фиксируем старт генераторов; отдельный запуск с новым seed даст другой эксперимент.
SEED = 2026
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

## Этап 1. Данные и нормализация

Изображение CIFAR-10 имеет форму `(3, 32, 32)`, пачка — `(B, 3, 32, 32)`. `ToTensor()` переводит байты в числа [0, 1], затем преобразование $x\mapsto2x-1$ переносит их в [-1, 1]. Это удобно для гауссовского зашумления. Обратное преобразование для показа: $(x+1)/2$.

Здесь метки классов `_` игнорируются: сеть генерирует без условия на класс. Данные загружаются при `download=True`; описание параметров — в [документации CIFAR10](https://docs.pytorch.org/vision/stable/generated/torchvision.datasets.CIFAR10.html).

**Сделайте:** запустите ячейку и проверьте, что показаны обычные цветные картинки. Обратите внимание на `permute(1, 2, 0)`: Matplotlib ждёт `(H, W, C)`, а PyTorch работает с `(C, H, W)`. Уменьшение `batch_size` снижает расход памяти, но не меняет число объектов в датасете.

In [ ]:
T = 1000  # Число уровней диффузии, не количество эпох.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
img_dim = 3 * 32 * 32  # 3072 координаты для MLP.
batch_size = 128

# CIFAR10: 32x32 RGB -> 3072 dim
transform = transforms.Compose([
    transforms.ToTensor(),  # [0, 1]
    transforms.Lambda(lambda x: x * 2. - 1.)  # [-1, 1]
])

train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
# Перемешиваем изображения при каждом проходе обучения.
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
# Одна пачка для знакомства с данными и демонстрации прямого процесса.
images, labels = next(iter(train_loader))

# Возьмём первые 50
images_50 = images[:50]
labels_50 = labels[:50]

def denormalize(img_tensor):
    return img_tensor * 0.5 + 0.5  # Обратная нормализация из [-1,1] → [0,1]

fig, axs = plt.subplots(5, 10, figsize=(8, 4))
for i in range(50):
    img = denormalize(images_50[i])
    axs[i // 10, i % 10].imshow(img.permute(1, 2, 0).numpy())  # CxHxW → HxWxC
    axs[i // 10, i % 10].axis('off')
plt.tight_layout()
plt.show()

## Этап 2. Расписание шума

Один прямой переход задаётся так:
$$q(x_t\mid x_{t-1})=\mathcal N(\sqrt{1-\beta_t}\,x_{t-1},\beta_tI).$$
Введём $\alpha_t=1-\beta_t$ и $\bar\alpha_t=\prod_{s=1}^{t}\alpha_s$. В следующей ячейке `cumprod` считает именно **произведение**, а не сумму.

**Как читать:** `beta` — дисперсия нового шума; его стандартное отклонение — `sqrt(beta)`. `sqrt(alphas_cumprod)` — множитель перед чистым изображением после накопленного зашумления. Все массивы должны находиться на том же устройстве, что изображения.

Линейное расписание от `1e-4` до `0.02` на 1000 шагов — конкретный учебный выбор. Если уменьшить `T`, сохранив эти границы, на последнем шаге может остаться слишком много сигнала. Тогда предположение старта генерации из $N(0,I)$ ухудшится. После изменения расписания модель нужно обучать заново с тем же расписанием, которое будет при генерации. Формулы прямого процесса: [DDPM, уравнения (2) и (4)](https://arxiv.org/pdf/2006.11239).

In [ ]:
T = 1000  # Математические уровни 1..T хранятся по индексам 0..T-1.
beta_start = 1e-4
beta_end = 0.02

# Создаём расписание один раз на том же устройстве, что и модель.
betas = torch.linspace(beta_start, beta_end, T, device=device)
alphas = 1.0 - betas
alphas_cumprod = torch.cumprod(alphas, dim=0)  # Произведение alpha от первого уровня до текущего.

# Эти коэффициенты позволяют получить любой уровень шума напрямую из чистой картинки.
sqrt_alphas_cumprod = torch.sqrt(alphas_cumprod)
sqrt_one_minus_alphas_cumprod = torch.sqrt(1.0 - alphas_cumprod)
assert torch.all((betas > 0) & (betas < 1))

## Этап 3. Получаем $x_t$ сразу из $x_0$

Последовательно добавлять шум t раз для обучения не требуется. Сумма независимых гауссовских приращений остаётся гауссовской, и можно использовать:
$$x_t=\sqrt{\bar\alpha_t}\,x_0+\sqrt{1-\bar\alpha_t}\,\epsilon,\qquad\epsilon\sim\mathcal N(0,I).$$

**Соответствие коду:** `sqrt_alpha * x_0` — ослабленный сигнал; `sqrt_one_minus_alpha * noise` — накопленный шум. У каждого изображения может быть свой уровень t. Массив коэффициентов формы `(B,)` преобразуется в `(B,1,1,1)`, чтобы один коэффициент применился ко всем пикселям соответствующей картинки. Это называется распространением по размерностям (*broadcasting*).

**Сделайте:** сравните несколько уровней. Ниже для разных уровней выбирается новый шум, то есть показаны выборки из маргинальных распределений $q(x_t\mid x_0)$, а не одна согласованная марковская траектория. Для визуального сравнения с одним направлением шума можно вынести создание `noise` за цикл, но это тоже не станет прямой марковской траекторией.

**Для отображения** переводим значения через `(x_t + 1)/2` и ограничиваем [0, 1]. Само состояние `x_t` при обучении не обрезаем: иначе нарушим гауссовскую формулу. График иллюстрирует разрушение структуры; численный контроль формулы выполняется тестами.

In [ ]:
def q_sample(x_0, t, noise=None):
    """Прямая выборка q(x_t | x_0).

    x_0: (B, C, H, W) или (B, dim); t: (B,), целые индексы 0..T-1.
    noise: стандартный нормальный шум той же формы, что x_0.
    Расписание, x_0 и t должны находиться на одном устройстве.
    """
    if noise is None:
        noise = torch.randn_like(x_0)
    batch_size = t.shape[0]
    # Один коэффициент для всех координат каждого объекта: (B, 1, 1, 1) для RGB.
    shape = [batch_size] + [1] * (x_0.dim() - 1)
    signal_scale = sqrt_alphas_cumprod[t].view(shape)
    noise_scale = sqrt_one_minus_alphas_cumprod[t].view(shape)
    return signal_scale * x_0 + noise_scale * noise

num_images = 5
timesteps = [0, 49, 99, 199, 499, 999]
fig, axs = plt.subplots(len(timesteps), num_images, figsize=(10, 12))
for row, time_index in enumerate(timesteps):
    x_0 = images[:num_images].to(device)
    t_tensor = torch.full((num_images,), time_index, device=device, dtype=torch.long)
    noise = torch.randn_like(x_0)
    x_t = q_sample(x_0, t_tensor, noise)
    # Только копия для визуализации: состояния процесса и обучения не обрезаем.
    displayed = ((x_t + 1) / 2).clamp(0, 1).cpu()
    for col in range(num_images):
        axs[row, col].imshow(displayed[col].permute(1, 2, 0).numpy())
        axs[row, col].axis('off')
        if col == 0:
            axs[row, col].set_title(f'Уровень {time_index + 1}', fontsize=10)
fig.suptitle('Прямой процесс DDPM: от изображения к шуму')
fig.tight_layout(rect=(0, 0, 1, 0.97))
plt.show()

## Этап 4. Чему учится сеть

Для каждого примера берём случайный уровень шума, генерируем известный шум и создаём зашумлённое изображение. Сеть получает $(x_t,t)$ и предсказывает $\epsilon_\theta(x_t,t)$.

$$L_{\mathrm{simple}}=\mathbb E_{x_0,t,\epsilon}\left[\|\epsilon-\epsilon_\theta(x_t,t)\|^2\right].$$

`F.mse_loss` усредняет квадрат ошибки по всем элементам пачки, включая пиксели и каналы. Поэтому численное значение отличается постоянным масштабом от записи через сумму квадратов, но оптимизирует ту же задачу при фиксированных размерностях.

Сеть не видит чистую картинку напрямую: целевое значение — **шум, который мы сами добавили**. Для каждого нового прохода уровни и шум выбираются заново. В таком обучении не нужно хранить все 1000 зашумлённых версий каждого изображения.

Эта упрощённая MSE-задача связана с вариационной границей, но не равна всему отрицательному логарифму правдоподобия. См. [DDPM, раздел 3.4 и уравнение (14)](https://arxiv.org/pdf/2006.11239).

### Этап 4A. Обучение DDPM с многослойным перцептроном (MLP)

### Как устроена MLP и как читать цикл обучения

1. `time_mlp` превращает скалярный индекс времени в вектор из 256 чисел.
2. Изображение разворачивается в `(B,3072)`, затем к нему добавляется вектор времени.
3. Основная MLP возвращает 3072 значений предсказанного шума.
4. `optimizer.zero_grad()` очищает прошлые градиенты, `loss.backward()` вычисляет новые, `optimizer.step()` обновляет веса.
5. Потери суммируются с весом размера пачки, чтобы последняя неполная пачка не исказила среднее за эпоху.

**Сделайте:** для проверки сначала установите `num_epochs=1`. Это всё равно полный проход по датасету; для короткого отладочного запуска можно заранее использовать `torch.utils.data.Subset`. Затем верните исходный датасет и увеличьте число эпох. Наблюдайте, уменьшается ли ошибка и меняются ли изображения после переобучения.

Здесь в MLP времени подаётся сырой индекс 0…999. Возможный следующий эксперимент — нормированный индекс или синусоидальное кодирование. Делать такую замену только при генерации нельзя: представление времени при обучении и использовании должно совпадать.

MLP не использует пространственную структуру изображения специально; это ограничивает качество. Пять эпох — настройка примера, а не гарантия узнаваемых изображений.

In [ ]:
import torch.nn as nn

class MLPDiffusion(nn.Module):
    def __init__(self, img_dim, time_dim=256):
        super().__init__()
        # Кодируем уровень шума отдельной небольшой сетью.
        self.time_mlp = nn.Sequential(
            nn.Linear(1, time_dim),
            nn.ReLU(),
            nn.Linear(time_dim, time_dim)
        )
        self.model = nn.Sequential(
            nn.Linear(img_dim + time_dim, 1024),
            nn.ReLU(),
            nn.Linear(1024, 1024),
            nn.ReLU(),
            nn.Linear(1024, img_dim)
        )

    def forward(self, x, t):
        t_embed = self.time_mlp(t.unsqueeze(-1).float())  # [B, time_dim]
        x_flat = x.view(x.size(0), -1)                    # [B, 3072]
        x = torch.cat([x_flat, t_embed], dim=1)           # [B, 3072 + time_dim]
        return self.model(x)                              # [B, 3072]


model = MLPDiffusion(img_dim).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
loss_history = []

num_epochs = 5

for epoch in range(num_epochs):
    epoch_loss = 0
    model.train()  # Режим обучения.
    for x_0, _ in train_loader:
        x_0 = x_0.to(device)
        # Независимый случайный уровень для каждой картинки, равномерно по 0..T-1.
        t = torch.randint(0, T, (x_0.size(0),), device=device).long()
        # Это целевой шум: сохраняем его, чтобы посчитать ошибку предсказания.
        noise = torch.randn_like(x_0)
        x_t = q_sample(x_0, t, noise)

        # Предсказание шума
        noise_pred = model(x_t, t)
        loss = F.mse_loss(noise_pred, noise.view(x_0.shape[0], -1))

        optimizer.zero_grad()  # Не накапливаем градиенты предыдущей пачки.
        loss.backward()  # Производные ошибки по обучаемым параметрам.
        optimizer.step()  # Один шаг оптимизатора Adam.

        # Взвешиваем по фактическому размеру пачки, включая последнюю неполную.
        epoch_loss += loss.item() * x_0.size(0)

    avg_epoch_loss = epoch_loss / len(train_loader.dataset)
    loss_history.append(avg_epoch_loss)
    print(f"Эпоха {epoch + 1}/{num_epochs} — Ошибка: {avg_epoch_loss:.6f}")

# Сохраняем веса; настройки расписания следует хранить вместе с экспериментом.
torch.save(model.state_dict(), "ddpm_mlp.pth")
print("Модель сохранена в файл ddpm_mlp.pth")

plt.figure(figsize=(8, 4))
plt.plot(loss_history, label="Ошибка MSE на обучении")
plt.xlabel("Эпоха")
plt.ylabel("Ошибка")
plt.title("Обучение DDPM: MLP")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

## Этап 4B. Обучение DDPM с простой свёрточной сетью (CNN)

### Свёрточный вариант: формы и ограничения

`ConvDiffusion` сохраняет форму `(B,3,32,32)` на выходе. Из временного вектора используется **только первая компонента**: она расширяется до постоянной карты `(B,1,32,32)` и добавляется к RGB как четвёртый канал. Остальные выходные компоненты временного слоя не участвуют в предсказании — это упрощение исходного примера.

Три свёртки 3 × 3 с `padding=1` сохраняют пространственное разрешение. Здесь нет сжатия, восстановления разрешения и skip-связей, поэтому название **U-Net** для этой реализации было неточным. При сравнении с публикациями учитывайте, что это значительно более простая сеть.

Цель обучения та же, что у MLP. Отличается форма выхода: свёрточная сеть сразу выдаёт изображение шума, и разворачивать целевой шум для MSE не требуется.

**Эксперимент:** сравнивайте модели при одинаковом расписании и сопоставимом бюджете. В исходном примере MLP обучается 5 эпох, CNN — 30; сравнение таких результатов не изолирует влияние архитектуры. Сначала задайте одинаковое число эпох и фиксируйте также время обучения.

In [ ]:
import torch.nn as nn

class ConvDiffusion(nn.Module):
    def __init__(self, in_channels=3, time_dim=128):
        super().__init__()
        # Кодируем уровень шума отдельной небольшой сетью.
        self.time_mlp = nn.Sequential(
            nn.Linear(1, time_dim),
            nn.ReLU(),
            nn.Linear(time_dim, time_dim)
        )
        self.conv = nn.Sequential(
            nn.Conv2d(in_channels + 1, 64, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 64, 3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, in_channels, 3, padding=1)
        )

    def forward(self, x, t):
        t_embed = self.time_mlp(t.unsqueeze(-1).float())  # [B, time_dim]
        # Расширяем временные признаки до размера картинки; ниже используем только первый.
        t_map = t_embed[:, :, None, None].expand(-1, -1, x.size(2), x.size(3))
        x = torch.cat([x, t_map[:, 0:1, :, :]], dim=1)  # Добавляем один канал с информацией о времени.
        return self.conv(x)

model = ConvDiffusion().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
loss_history = []

num_epochs = 30  # Для проверки можно начать с 1; это не гарантия качества.

for epoch in range(num_epochs):
    epoch_loss = 0
    model.train()
    for step, (x_0, _) in enumerate(train_loader):
        x_0 = x_0.to(device)
        # Независимый случайный уровень для каждой картинки, равномерно по 0..T-1.
        t = torch.randint(0, T, (x_0.size(0),), device=device).long()
        # Это целевой шум: сохраняем его, чтобы посчитать ошибку предсказания.
        noise = torch.randn_like(x_0)

        x_t = q_sample(x_0, t, noise)
        noise_pred = model(x_t, t)

        loss = F.mse_loss(noise_pred, noise)

        optimizer.zero_grad()  # Не накапливаем градиенты предыдущей пачки.
        loss.backward()  # Производные ошибки по обучаемым параметрам.
        optimizer.step()  # Один шаг оптимизатора Adam.

        # Взвешиваем по фактическому размеру пачки, включая последнюю неполную.
        epoch_loss += loss.item() * x_0.size(0)

        if step % 100 == 0:
            print(f"Эпоха {epoch+1}, Пачка {step}, Ошибка: {loss.item():.4f}")

    avg_epoch_loss = epoch_loss / len(train_loader.dataset)
    loss_history.append(avg_epoch_loss)
    print(f"Эпоха {epoch + 1}/{num_epochs} — Средняя ошибка: {avg_epoch_loss:.6f}")

# Сохраняем веса; настройки расписания следует хранить вместе с экспериментом.
torch.save(model.state_dict(), "ddpm_conv.pth")
print("Модель сохранена в файл ddpm_conv.pth")

plt.figure(figsize=(8, 4))
plt.plot(loss_history, label="Ошибка MSE на обучении")
plt.xlabel("Эпоха")
plt.ylabel("Ошибка")
plt.title("Обучение DDPM: CNN")
plt.grid(True)
plt.legend()
plt.tight_layout()
plt.show()

## Этап 5. Подготовка к генерации изображений

### Выбираем обученную модель

Выполните хотя бы один из тренировочных блоков, затем выберите `MODEL_KIND`. MLP загружает `ddpm_mlp.pth`, CNN — `ddpm_conv.pth`. Архитектура и её параметры должны совпадать с теми, для которых сохранены веса.

`map_location=device` переносит веса на текущее устройство. `weights_only=True` соответствует загрузке обычного словаря тензоров. `model.eval()` включает режим использования сети; отключение вычисления градиентов задаётся отдельно через `@torch.no_grad()` в функциях генерации.

В этих простых сетях нет dropout и batch normalization, но вызов `eval()` сохраняет правильный порядок работы при дальнейшем расширении архитектуры. Рядом с весами сохраняйте настройки T, beta, нормализации и архитектуры: один `state_dict` не содержит полного описания эксперимента.

In [ ]:
# Выберите архитектуру, для которой выше были обучены и сохранены веса.
MODEL_KIND = 'mlp'  # 'mlp' или 'conv'
if MODEL_KIND == 'mlp':
    model = MLPDiffusion(img_dim).to(device)
    checkpoint_path = 'ddpm_mlp.pth'
elif MODEL_KIND == 'conv':
    model = ConvDiffusion().to(device)
    checkpoint_path = 'ddpm_conv.pth'
else:
    raise ValueError("MODEL_KIND должен быть 'mlp' или 'conv'.")

# Используйте веса от того же расписания и той же архитектуры.
model.load_state_dict(torch.load(checkpoint_path, map_location=device, weights_only=True))
model.eval()
print(f'Загружена модель {MODEL_KIND}: {checkpoint_path}')

### Проверка расписания перед генерацией

Следующий график показывает накопленное ослабление сигнала. На последних шагах $\bar\alpha_t$ должно быть мало, чтобы начальный гауссовский шум обратной цепочки соответствовал прямому процессу. График не подтверждает, что сеть уже обучена: он проверяет только расписание.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
levels = torch.arange(1, T + 1)
ax.plot(levels, alphas_cumprod.detach().cpu(), label='Накопленный коэффициент сигнала')
ax.plot(levels, (1 - alphas_cumprod).detach().cpu(), label='Накопленная дисперсия шума')
ax.set(xlabel='Математический уровень t', ylabel='Значение', title='Расписание прямого процесса')
ax.legend()
fig.tight_layout()
plt.show()
print(f'Коэффициент сигнала на последнем уровне: {alphas_cumprod[-1].item():.6g}')

## Этап 6. Один обратный шаг

Среднее обратного перехода параметризуется предсказанным шумом:
$$\mu_\theta(x_t,t)=\frac1{\sqrt{\alpha_t}}
\left(x_t-\frac{\beta_t}{\sqrt{1-\bar\alpha_t}}\epsilon_\theta(x_t,t)\right).$$

Затем получаем выборку $x_{t-1}=\mu_\theta(x_t,t)+\sigma_tz$, где $z\sim N(0,I)$. Ниже выбран вариант $\sigma_t^2=\beta_t$. Это один из фиксированных вариантов в [DDPM, раздел 3.2](https://arxiv.org/pdf/2006.11239).

Не путайте его с точной дисперсией **условного** прямого апостериорного распределения:
$$\tilde\beta_t=\beta_t\frac{1-\bar\alpha_{t-1}}{1-\bar\alpha_t},\qquad
q(x_{t-1}\mid x_t,x_0).$$

В ней дополнительно известен $x_0$. При генерации чистая картинка неизвестна, и её информацию приближает сеть. В статье рассматриваются оба фиксированных выбора дисперсии; в этом ноутбуке последовательно используется `beta_t`.

**Читайте строку за строкой:** извлечь коэффициенты → предсказать шум → привести его к форме `x_t` → вычислить среднее → добавить новый шум, кроме последнего шага. `t != 0` относится к Python-индексу: на финальном переходе нужен результат среднего без дополнительной случайности.

In [ ]:
@torch.no_grad()
def p_sample(model, x_t, t):
    """
    Один шаг обратного диффузионного процесса
    x_t: [B, C, H, W] или [B, dim] для MLP
    t: [B] — Tensor, значения в диапазоне [0, T)
    """
    batch_size = x_t.shape[0]
    shape = [batch_size] + [1] * (x_t.dim() - 1)  # [B, 1, 1, 1]

    # Индексация коэффициентов по каждому элементу батча
    beta_t = betas[t].reshape(shape)
    alpha_t = alphas[t].reshape(shape)
    alpha_cumprod_t = alphas_cumprod[t].reshape(shape)
    sqrt_one_minus_alpha_cumprod_t = torch.sqrt(1 - alpha_cumprod_t)
    sqrt_recip_alpha_t = torch.rsqrt(alpha_t)  # 1 / sqrt(alpha)

    # Предсказание шума
    # MLP возвращает плоский шум, CNN — изображение; согласуем с формой состояния.
    eps_theta = model(x_t, t).reshape_as(x_t)

    # Среднее обратного перехода: ослабляем предсказанный шум и делим на sqrt(alpha_t).
    model_mean = sqrt_recip_alpha_t * (x_t - (beta_t / sqrt_one_minus_alpha_cumprod_t) * eps_theta)

    # Добавляем шум (кроме финального шага)
    noise = torch.randn_like(x_t)
    nonzero_mask = (t != 0).float().reshape(shape)  # чтобы не добавлять шум на t=0
    # Выбранная дисперсия обратного перехода — beta_t, не апостериорная beta_tilde.
    sample = model_mean + nonzero_mask * torch.sqrt(beta_t) * noise

    return sample

## Этап 7. Полная обратная цепочка

Стартуем с новой независимой выборки $x_T\sim N(0,I)$ и вызываем обратный шаг для индексов `T-1,...,0`. Здесь число вызовов сети равно T, а не числу эпох обучения.

Все картинки на одной итерации получают один уровень t, но каждая — собственный шум. Начальный шум и случайные приращения объясняют, почему разные запуски создают разные изображения.

Просто пропустить часть индексов в этом цикле нельзя: формула рассчитана на соседние уровни исходного расписания. Для генерации меньшим числом шагов потребуется согласованный метод семплирования, а не только изменение `range`.

В отличие от стационарного Ланжевена из Lab One, здесь используется последовательность уровней шума и обученная функция. Схемы связаны математически, но не взаимозаменяемы без вывода коэффициентов.

In [ ]:
@torch.no_grad()
def p_sample_loop(model, shape, device):
    """Полная обратная цепочка, T вызовов сети. shape: (B, C, H, W) или (B, dim) для MLP."""
    x = torch.randn(shape, device=device)  # Новый исходный шум, не картинка из датасета.
    for time_index in reversed(range(T)):
        # На этом шаге у всей пачки общий уровень шума.
        t_batch = torch.full((x.shape[0],), time_index, device=device, dtype=torch.long)
        x = p_sample(model, x, t_batch)
    return x

## Этап 8. Показываем результат

Генерация поддерживает обе модели: состояние имеет форму изображения, а плоский выход MLP преобразуется обратно внутри `p_sample`. Ограничение диапазона и перевод в [0,1] происходят только для конечного отображения.

**Оцените результат:** видна ли структура; разнообразны ли примеры; не повторяются ли артефакты; улучшились ли они после более долгого обучения при том же начальном seed? Четыре картинки и тренировочная MSE не заменяют полноценную оценку качества.

Если виден только шум, проверьте: загрузились ли нужные веса; совпадает ли расписание с тренировочным; не перепутаны ли формы MLP/CNN; не обрезался ли `x_t` при обучении; хватает ли модели и длительности обучения. Успешное выполнение кода ещё не означает, что модель хорошо оценила распределение данных.

In [ ]:
@torch.no_grad()
def generate_images(model, num_images=4):
    # Обе модели получают состояние в форме изображения.
    samples = p_sample_loop(model, shape=(num_images, 3, 32, 32), device=device)
    # Ограничиваем только конечный результат для показа.
    images = samples.view(-1, 3, 32, 32).clamp(-1, 1)
    images = (images + 1) / 2  # [-1, 1] → [0, 1]
    return images.cpu()

def show_images(images):
    plt.figure(figsize=(12, 3))
    for i in range(images.shape[0]):
        plt.subplot(1, images.shape[0], i + 1)
        plt.imshow(images[i].permute(1, 2, 0).numpy())
        plt.axis("off")
    plt.tight_layout()
    plt.show()

# Генерация и визуализация из нового случайного шума.
generated_imgs = generate_images(model, num_images=4)
show_images(generated_imgs)

## Связь с Lab One: шум, score и СДУ

Для фиксированного чистого изображения условный score гауссовского зашумления можно вывести непосредственно:
$$\nabla_{x_t}\log q(x_t\mid x_0)
=-\frac{x_t-\sqrt{\bar\alpha_t}x_0}{1-\bar\alpha_t}
=-\frac{\epsilon}{\sqrt{1-\bar\alpha_t}}.$$
Сеть, оптимальная по MSE, оценивает условное среднее шума при известных $(x_t,t)$. Поэтому её масштабированное предсказание даёт оценку **маргинальной** скор-функции:
$$s_\theta(x_t,t)\approx-\frac{\epsilon_\theta(x_t,t)}{\sqrt{1-\bar\alpha_t}}.$$
Это объясняет, почему обучение предсказанию шума связано с направлением движения к более вероятным точкам. Условный score при известном $x_0$ и score распределения всех зашумлённых данных — разные объекты; условное усреднение связывает их.

В пределе малых прямых шагов дискретное зашумление соответствует варианту СДУ с дрейфом, пропорциональным $-x$, и зависящей от времени диффузией. Для строгого вывода обратного СДУ и связанного ОДУ см. [Song et al., Score-Based Generative Modeling through SDEs](https://arxiv.org/abs/2011.13456). Формулы Ланжевена из Lab One полезны как подготовка, но подстановка их без изменения коэффициентов не реализует обратный DDPM.

## Вопросы для самостоятельного отчёта

1. Почему `beta_t` и `sqrt(beta_t)` играют разные роли?
2. Почему можно получить $x_t$ одним вызовом `q_sample`? Как отличается набор таких картинок от одной марковской траектории?
3. Почему при каждом обучающем шаге выбираются новый t и новый шум?
4. Что сломается, если сравнивать выход MLP `(B,3072)` с шумом `(B,3,32,32)` без изменения формы?
5. Почему нельзя использовать другое расписание только при генерации?
6. Почему последний обратный переход выполняется без дополнительного шума?
7. Чем простая CNN здесь отличается от настоящей U-Net?
8. Каких доказательств качества не даёт один график тренировочной ошибки?

**Шаблон записи эксперимента:** архитектура, seed, T, границы beta, batch size, learning rate, эпохи, устройство, время, кривая потерь и выборка изображений при фиксированном начальном seed. Изменяйте только один фактор за запуск. Сравнивайте одинаковую нормализацию и бюджет обучения.

## Что было исправлено при добавлении пояснений

- Удалено повторное создание расписания на CPU; теперь оно согласовано с `device`.
- В прямой визуализации согласованы устройства; для показа корректно обращается нормализация.
- Свёрточная модель правильно названа CNN; её устройство и ограничения описаны явно.
- Можно выбрать контрольную точку MLP или CNN, а генерация согласует форму предсказанного шума с состоянием.
- Английские заголовки и подписи заменены русскими; добавлены формулы, комментарии и проверки расписания.

## Литература и порядок чтения

- [Ho, Jain, Abbeel — Denoising Diffusion Probabilistic Models (2020)](https://arxiv.org/abs/2006.11239): первоисточник алгоритма; сопоставьте прямой процесс, обучение и генерацию с алгоритмами 1–2.
- [Lilian Weng — What are Diffusion Models?](https://lilianweng.github.io/posts/2021-07-11-diffusion-models/): дополнительное чтение с развёрнутыми выводами. [Главная страница блога](https://lilianweng.github.io/).
- [Song et al. — Score-Based Generative Modeling through SDEs](https://arxiv.org/abs/2011.13456): следующий шаг после понимания DDPM и Lab One.
- [Goodfellow, Bengio, Courville — Deep Learning](https://www.deeplearningbook.org/): главы 3 (вероятность), 6 (прямые сети), 8 (оптимизация), 20 (глубокие генеративные модели). Книга даёт базу; это не отдельное руководство по современному DDPM.
- [Øksendal — Stochastic Differential Equations](https://link.springer.com/book/10.1007/978-3-642-14394-6): строгая теория СДУ.
- [Higham — An Algorithmic Introduction to Numerical Simulation of SDEs](https://epubs.siam.org/doi/10.1137/S0036144500378302): численные методы и сходимость.
- [MIT: курс 2026](https://diffusion.csail.mit.edu/2026/): лекции, конспект и лабораторные.